In [4]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [5]:
df = pd.read_csv("../data/raw/BangaloreZomatoData.csv")
df.head()

,Name,URL,Cuisines,Area,Timing,Full_Address,PhoneNumber,IsHomeDelivery,isTakeaway,isIndoorSeating,isVegOnly,Dinner Ratings,Dinner Reviews,Delivery Ratings,Delivery Reviews,KnownFor,PopularDishes,PeopleKnownFor,AverageCost
0,Sri Udupi Park,https://www.zomato.com/bangalore/sri-udupi-par...,"South Indian, North Indian, Chinese, Street Fo...","Indiranagar, Bangalore",7am – 11pm (Today),"273, Monalisa, 6th Main, 100 Feet Road, Indira...",+919945977774,1,1,1,1,4.0,462,4.1,16000,NaN,"Filtered Coffee, Sambhar, Pav Bhaji, Gobi Manc...","Economical, Prompt Service, Hygiene, Quality F...",450
1,Meghana Foods,https://www.zomato.com/bangalore/meghana-foods...,"Biryani, Andhra, North Indian, Seafood","Indiranagar, Bangalore",Opens at 6:30pm,"544, First Floor, CMH Road, Near Indiranagar M...",+918041135050,1,1,1,0,4.3,1654,4.3,28600,Spicy Chicken Biryani,"Authentic Hyderabadi Biryani, Paneer Biryani, ...","Boneless Chicken Biryani, Ample Seating Area, ...",700
2,Donne Biriyani House,https://www.zomato.com/bangalore/donne-biriyan...,Biryani,"Indiranagar, Bangalore",11am – 11pm (Today),"8/ 9, 17th F Cross, 2nd Stage, Indiranagar, Ba...",+918861564169,1,1,1,0,3.9,411,3.5,33200,NaN,NaN,"Great Recommendations, Nice Taste, Great Ambia...",300
3,Domino's Pizza,https://www.zomato.com/bangalore/dominos-pizza...,"Pizza, Fast Food, Desserts","Indiranagar, Bangalore",10:57am – 12midnight (Today),"308, 2nd Stage, 100 Feet Road, Indiranagar, Ba...",+919916465787,1,1,1,0,2.4,422,4.4,8205,NaN,"Barbeque Chicken Pizza, Choco Lava Cake, White...","Value for Money, Packaging, Staff, Ambience, Food",400
4,KFC,https://www.zomato.com/bangalore/kfc-indiranagar,"Burger, Fast Food, Biryani, Desserts, Beverages","Indiranagar, Bangalore",11am – 11pm (Today),"38/1A, CMH Road, Indiranagar, Bangalore",+919513700040,1,1,1,0,2.8,673,4.0,9148,NaN,"Fiery Chicken, Chicken Popcorn, Rice Bowl, Wings","Elegantly Decorated, Great Recommendations, Vi...",400


In [6]:
print(f"Raw shape: {df.shape}")

Raw shape: (8923, 19)


In [7]:
print("\nNull counts (raw):")
print(df.isnull().sum())
 
print("\n'-' counts in rating columns:")
print("Dinner Ratings:", (df["Dinner Ratings"] == "-").sum())
print("Delivery Ratings:", (df["Delivery Ratings"] == "-").sum())


Null counts (raw):
Name                   0
URL                    0
Cuisines               0
Area                   0
Timing              3103
Full_Address           0
PhoneNumber            0
IsHomeDelivery         0
isTakeaway             0
isIndoorSeating        0
isVegOnly              0
Dinner Ratings         0
Dinner Reviews         0
Delivery Ratings       0
Delivery Reviews       0
KnownFor            8665
PopularDishes       7388
PeopleKnownFor      5439
AverageCost            0
dtype: int64

'-' counts in rating columns:
Dinner Ratings: 5325
Delivery Ratings: 1131


In [8]:
before = len(df)
df = df.drop_duplicates(subset=["Name", "Full_Address"]).reset_index(drop=True)
print(f"\nDropped {before - len(df)} duplicate rows")


Dropped 2 duplicate rows


In [9]:
for col in ["Dinner Ratings", "Delivery Ratings"]:
    flag_col = col.replace(" Ratings", "_Rating_Unavailable")
    df[flag_col] = (df[col] == "-").astype(int)
    df[col] = pd.to_numeric(df[col].replace("-", np.nan), errors="coerce")

In [10]:
suspicious = df[(df["Dinner Ratings"].notna()) & (df["Dinner Reviews"] == 0)]
print(f"\nRows with a Dinner rating but 0 Dinner reviews: {len(suspicious)}")


Rows with a Dinner rating but 0 Dinner reviews: 47


In [11]:
df["Timing"] = df["Timing"].fillna("Not Listed")

In [12]:
for col in ["KnownFor", "PopularDishes", "PeopleKnownFor"]:
    df[col] = df[col].fillna("None")

In [13]:
print("\nAverageCost before outlier handling:")
print(df["AverageCost"].describe())
q1, q3 = df["AverageCost"].quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 3 * iqr  # wide fence: 3xIQR, since cost legitimately varies a lot by venue type
outliers = df[df["AverageCost"] > upper_fence]
print(f"Extreme cost outliers (> {upper_fence:.0f} INR): {len(outliers)} rows — kept, but flagged")
df["IsCostOutlier"] = (df["AverageCost"] > upper_fence).astype(int)


AverageCost before outlier handling:
count    8921.000000
mean      340.273512
std       308.356437
min        50.000000
25%       150.000000
50%       250.000000
75%       400.000000
max      4200.000000
Name: AverageCost, dtype: float64
Extreme cost outliers (> 1150 INR): 282 rows — kept, but flagged


In [14]:
bool_cols = ["IsHomeDelivery", "isTakeaway", "isIndoorSeating", "isVegOnly"]
for col in bool_cols:
    df[col] = df[col].astype(int)

In [15]:
df["Locality"] = df["Area"].str.split(",").str[0].str.strip()

In [16]:
df["Primary_Cuisine"] = df["Cuisines"].str.split(",").str[0].str.strip()

In [17]:
print("\nNull counts (post-clean):")
print(df.isnull().sum())


Null counts (post-clean):
Name                              0
URL                               0
Cuisines                          0
Area                              0
Timing                            0
Full_Address                      0
PhoneNumber                       0
IsHomeDelivery                    0
isTakeaway                        0
isIndoorSeating                   0
isVegOnly                         0
Dinner Ratings                 5323
Dinner Reviews                    0
Delivery Ratings               1130
Delivery Reviews                  0
KnownFor                          0
PopularDishes                     0
PeopleKnownFor                    0
AverageCost                       0
Dinner_Rating_Unavailable         0
Delivery_Rating_Unavailable       0
IsCostOutlier                     0
Locality                          0
Primary_Cuisine                   0
dtype: int64


In [18]:
df.to_csv("..\data\BangaloreZomatoData_clean.csv", index=False)
print(f"\nClean shape: {df.shape}")
print("Saved -> BangaloreZomatoData_clean.csv")


Clean shape: (8921, 24)
Saved -> BangaloreZomatoData_clean.csv


In [19]:
print("\nSummary statistics (numeric columns):")
df.describe()


Summary statistics (numeric columns):


,IsHomeDelivery,isTakeaway,isIndoorSeating,isVegOnly,Dinner Ratings,Dinner Reviews,Delivery Ratings,Delivery Reviews,AverageCost,Dinner_Rating_Unavailable,Delivery_Rating_Unavailable,IsCostOutlier
count,8921.000000,8921.000000,8921.000000,8921.000000,3598.000000,8921.000000,7791.000000,8921.000000,8921.000000,8921.000000,8921.000000,8921.000000
mean,0.997870,0.660128,0.442775,0.072301,3.645525,157.141240,3.879143,2015.138101,340.273512,0.596682,0.126667,0.031611
std,0.046103,0.473692,0.496742,0.259001,0.474745,731.912485,0.320954,5524.975465,308.356437,0.490591,0.332619,0.174971
min,0.000000,0.000000,0.000000,0.000000,2.000000,0.000000,2.400000,0.000000,50.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,3.300000,0.000000,3.700000,42.000000,150.000000,0.000000,0.000000,0.000000
50%,1.000000,1.000000,0.000000,0.000000,3.700000,0.000000,3.900000,279.000000,250.000000,1.000000,0.000000,0.000000
75%,1.000000,1.000000,1.000000,0.000000,4.000000,43.000000,4.100000,1494.000000,400.000000,1.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,4.900000,26500.000000,4.700000,99600.000000,4200.000000,1.000000,1.000000,1.000000


In [20]:
print("\nTop 10 localities by restaurant count:")
df["Locality"].value_counts().head(10)


Top 10 localities by restaurant count:


Locality
Electronic City    673
Marathahalli       484
HSR                457
Whitefield         447
BTM                397
Indiranagar        292
JP Nagar           292
Sarjapur Road      284
Rajajinagar        237
New BEL Road       231
Name: count, dtype: int64

In [21]:
print("\nTop 10 cuisines by frequency:")
df["Primary_Cuisine"].value_counts().head(10) 


Top 10 cuisines by frequency:


Primary_Cuisine
North Indian    1619
Biryani          970
South Indian     958
Chinese          643
Fast Food        481
Bakery           444
Desserts         345
Ice Cream        313
Pizza            286
Street Food      266
Name: count, dtype: int64

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df["Dinner Ratings"].dropna(), bins=20, kde=True, ax=axes[0], color="#E74C3C")
axes[0].set_title("Distribution of Dinner Ratings")
sns.histplot(df["Delivery Ratings"].dropna(), bins=20, kde=True, ax=axes[1], color="#3498DB")
axes[1].set_title("Distribution of Delivery Ratings")
plt.tight_layout()
plt.savefig("../docs/plots/01_rating_distributions.png", dpi=150)

plt.close()

In [26]:
top_localities = df["Locality"].value_counts().head(10).index
plt.figure(figsize=(10, 6))
sns.boxplot(
    data=df[df["Locality"].isin(top_localities)],
    x="AverageCost", y="Locality",
    orient="h", palette="viridis"
)
plt.title("Average Cost by Locality (Top 10 Localities)")
plt.xlabel("Average Cost (INR)")
plt.tight_layout()
plt.savefig("../docs/plots/cost_by_locality.png", dpi=150)
plt.close()

In [ ]:
df.columns

In [ ]:
df[['Dinner Ratings', 'Dinner Reviews', 'Delivery Ratings',
       'Delivery Reviews']]